In [ ]:
import os
import requests
from tqdm import tqdm
from pathlib import Path


In [ ]:

#Configuration 
YEARS_TO_FETCH = [2020, 2021, 2022, 2023, 2024, 2025]
CVES_PER_YEAR_LIMIT = 500
BASE_URL = "https://raw.githubusercontent.com/espressif/esp-nvd-mirror/master/cve"
RAW_DATA_DIR = Path("data/01_raw_cve_json")



In [ ]:
#Main Execution

def fetch_cve_data():
    """
    Downloads and stores CVE data for the specified years.
    """
    print("Initiating CVE data download...")
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

    for year in YEARS_TO_FETCH:
        year_dir = RAW_DATA_DIR / str(year)
        year_dir.mkdir(exist_ok=True)
        print(f"\nProcessing year: {year}")

        for num in tqdm(range(1, CVES_PER_YEAR_LIMIT + 1), desc=f"Fetching CVEs for {year}"):
            cve_id = f"CVE-{year}-{num:04d}.json"
            url = f"{BASE_URL}/{year}/{cve_id}"
            file_path = year_dir / cve_id

            if file_path.exists():
                continue

            try:
                response = requests.get(url, timeout=10)
                if response.status_code == 200:
                    with open(file_path, "wb") as f:
                        f.write(response.content)
                elif response.status_code == 404:
                    pass
                else:
                    print(f"Warning: Skipped {cve_id} (Status: {response.status_code})")

            except requests.exceptions.RequestException as e:
                print(f"Error fetching {cve_id}: {e}")

    print("\nData acquisition finished.")
    print(f"Raw JSON data saved in: {RAW_DATA_DIR}")

if __name__ == "__main__":
    fetch_cve_data()